In [2]:
# ============================================================
# NOTEBOOK 02 — MAPPING & CLEANING
# Think of this like translating a document.
# IBM named columns their way. Workday wants them differently.
# We also drop useless columns and fix messy values.
# ============================================================

# Cell 1 — Import libraries
import pandas as pd

print("✅ Libraries loaded!")

✅ Libraries loaded!


In [4]:
# Cell 2 — Load the RAW data
# We always start from raw. Never edit the original.
# That's rule #1 in data migration — raw data is sacred.

df = pd.read_csv("../Data/raw_hr_data.csv")

print(f"📊 Loaded: {df.shape[0]} rows, {df.shape[1]} columns")

📊 Loaded: 1470 rows, 35 columns


In [6]:
# Cell 3 — Drop useless columns
# Remember from Notebook 01: these 3 columns never change.
# EmployeeCount = always 1
# Over18 = always Y
# StandardHours = always 80
# Useless columns = noise. Workday doesn't need them.

columns_to_drop = ["EmployeeCount", "Over18", "StandardHours"]

df = df.drop(columns=columns_to_drop)

print(f"🗑️ Dropped 3 useless columns.")
print(f"📊 Now: {df.shape[0]} rows, {df.shape[1]} columns")

🗑️ Dropped 3 useless columns.
📊 Now: 1470 rows, 32 columns


In [8]:
# Cell 4 — Rename columns to Workday-style names
# Workday uses snake_case with clear prefixes.
# Example: "MonthlyIncome" becomes "compensation_monthly_salary"
# This mapping dict is like a translation dictionary.
# Left side = old IBM name. Right side = new Workday name.

column_mapping = {
    "Age"                      : "worker_age",
    "Attrition"                : "employment_attrition_flag",
    "BusinessTravel"           : "travel_frequency",
    "DailyRate"                : "compensation_daily_rate",
    "Department"               : "org_department",
    "DistanceFromHome"         : "worker_commute_distance",
    "Education"                : "education_level_code",
    "EducationField"           : "education_field",
    "EmployeeNumber"           : "worker_id",
    "EnvironmentSatisfaction"  : "survey_environment_satisfaction",
    "Gender"                   : "worker_gender",
    "HourlyRate"               : "compensation_hourly_rate",
    "JobInvolvement"           : "survey_job_involvement",
    "JobLevel"                 : "org_job_level",
    "JobRole"                  : "org_job_role",
    "JobSatisfaction"          : "survey_job_satisfaction",
    "MaritalStatus"            : "worker_marital_status",
    "MonthlyIncome"            : "compensation_monthly_salary",
    "MonthlyRate"              : "compensation_monthly_rate",
    "NumCompaniesWorked"       : "career_companies_worked_count",
    "OverTime"                 : "work_overtime_flag",
    "PercentSalaryHike"        : "compensation_salary_hike_pct",
    "PerformanceRating"        : "performance_rating_code",
    "RelationshipSatisfaction" : "survey_relationship_satisfaction",
    "StockOptionLevel"         : "compensation_stock_option_level",
    "TotalWorkingYears"        : "career_total_years",
    "TrainingTimesLastYear"    : "training_sessions_last_year",
    "WorkLifeBalance"          : "survey_work_life_balance",
    "YearsAtCompany"           : "tenure_total_years",
    "YearsInCurrentRole"       : "tenure_current_role_years",
    "YearsSinceLastPromotion"  : "tenure_years_since_promotion",
    "YearsWithCurrManager"     : "tenure_years_with_manager"
}

df = df.rename(columns=column_mapping)

print("✅ Columns renamed to Workday-style.")
print(df.columns.tolist())

✅ Columns renamed to Workday-style.
['worker_age', 'employment_attrition_flag', 'travel_frequency', 'compensation_daily_rate', 'org_department', 'worker_commute_distance', 'education_level_code', 'education_field', 'worker_id', 'survey_environment_satisfaction', 'worker_gender', 'compensation_hourly_rate', 'survey_job_involvement', 'org_job_level', 'org_job_role', 'survey_job_satisfaction', 'worker_marital_status', 'compensation_monthly_salary', 'compensation_monthly_rate', 'career_companies_worked_count', 'work_overtime_flag', 'compensation_salary_hike_pct', 'performance_rating_code', 'survey_relationship_satisfaction', 'compensation_stock_option_level', 'career_total_years', 'training_sessions_last_year', 'survey_work_life_balance', 'tenure_total_years', 'tenure_current_role_years', 'tenure_years_since_promotion', 'tenure_years_with_manager']


In [10]:
# Cell 5 — Standardize Yes/No flag columns
# Workday expects clean boolean-style values.
# "Yes" → 1, "No" → 0
# This makes it easier for systems to read. 
# 1 = true/yes, 0 = false/no — universal language.

flag_columns = ["employment_attrition_flag", "work_overtime_flag"]

for col in flag_columns:
    df[col] = df[col].map({"Yes": 1, "No": 0})
    print(f"✅ {col}: Yes→1, No→0")

✅ employment_attrition_flag: Yes→1, No→0
✅ work_overtime_flag: Yes→1, No→0


In [12]:
# Cell 6 — Standardize Gender values
# Systems want consistent values.
# "Male" → "M", "Female" → "F"
# Short codes = less storage, less typo risk.

df["worker_gender"] = df["worker_gender"].map({"Male": "M", "Female": "F"})

print("✅ Gender standardized: Male→M, Female→F")
print(df["worker_gender"].value_counts())

✅ Gender standardized: Male→M, Female→F
worker_gender
M    882
F    588
Name: count, dtype: int64


In [14]:
# Cell 7 — Standardize Travel Frequency values
# Original values are wordy and inconsistent.
# We clean them into short standard codes Workday expects.

travel_mapping = {
    "Non-Travel"                        : "NONE",
    "Travel_Rarely"                     : "RARE",
    "Travel_Frequently"                 : "FREQ"
}

df["travel_frequency"] = df["travel_frequency"].map(travel_mapping)

print("✅ Travel frequency standardized.")
print(df["travel_frequency"].value_counts())

✅ Travel frequency standardized.
travel_frequency
RARE    1043
FREQ     277
NONE     150
Name: count, dtype: int64


In [16]:
# Cell 8 — Add a worker_status column
# Workday needs an explicit employment status field.
# If attrition = 1 (they left), status = "Terminated"
# If attrition = 0 (still here), status = "Active"
# We DERIVE this new column from existing data.

df["worker_status"] = df["employment_attrition_flag"].map({1: "Terminated", 0: "Active"})

print("✅ worker_status column created.")
print(df["worker_status"].value_counts())

✅ worker_status column created.
worker_status
Active        1233
Terminated     237
Name: count, dtype: int64


In [18]:
# Cell 9 — Verify no new nulls were introduced
# When we use .map(), if a value wasn't in our dictionary,
# it becomes NaN (null). We must check we didn't break anything.

print("🔍 Checking for nulls after cleaning:\n")

null_check = df.isnull().sum()
problem_cols = null_check[null_check > 0]

if problem_cols.empty:
    print("✅ No nulls. All mappings worked correctly.")
else:
    print("⚠️ These columns have nulls — check your mapping:")
    print(problem_cols)

🔍 Checking for nulls after cleaning:

✅ No nulls. All mappings worked correctly.


In [20]:
# Cell 10 — Preview the cleaned data
# Let's look at a few rows to confirm everything looks right.
# Eyes on the data — always verify visually too.

print("👀 First 3 rows of cleaned data:\n")
df.head(3)

👀 First 3 rows of cleaned data:



,worker_age,employment_attrition_flag,travel_frequency,compensation_daily_rate,org_department,worker_commute_distance,education_level_code,education_field,worker_id,survey_environment_satisfaction,...,survey_relationship_satisfaction,compensation_stock_option_level,career_total_years,training_sessions_last_year,survey_work_life_balance,tenure_total_years,tenure_current_role_years,tenure_years_since_promotion,tenure_years_with_manager,worker_status
0,41,1,RARE,1102,Sales,1,2,Life Sciences,1,2,...,1,0,8,0,1,6,4,0,5,Terminated
1,49,0,FREQ,279,Research & Development,8,1,Life Sciences,2,3,...,4,1,10,3,3,10,7,1,7,Active
2,37,1,RARE,1373,Research & Development,2,2,Other,4,4,...,2,0,7,3,3,0,0,0,0,Terminated


In [22]:
# Cell 11 — Save cleaned file
# We save to a NEW file. Raw file stays untouched. Always.
# This is your cleaned, Workday-ready dataset.

output_path = "../Data/raw_hr_data.csv"

df.to_csv(output_path, index=False)

print(f"💾 Cleaned file saved!")
print(f"📊 Final shape: {df.shape[0]} rows, {df.shape[1]} columns")
print("✅ Notebook 02 complete. Move to 03_validation.ipynb next.")

💾 Cleaned file saved!
📊 Final shape: 1470 rows, 33 columns
✅ Notebook 02 complete. Move to 03_validation.ipynb next.
